# Environment probe

Checks, in order: the GPU, the system Python, then provisions the project's
Python 3.11 venv (scripts/setup_colab.sh, uv + pinned wheels), then verifies the
venv can import torch/mmcv/mmdet/mmrotate with CUDA, then tries to mount Drive.


In [ ]:
#@title 0 - Clone the repo, show the GPU and the system Python
import subprocess, sys
from pathlib import Path

REPO = Path('/content/ADA')
COMMIT = 'a2ecd69d3d487c3f639c3cec329e0a737454370e'

print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
print('system python', sys.version.split()[0], '->', sys.executable)
print('disk:', subprocess.run(['df', '-h', '/content'], capture_output=True, text=True).stdout.splitlines()[-1])

if not (REPO / 'scripts').is_dir():
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/Lion4467cat/dotav2-oriented-faster-rcnn-benchmark.git',
                    str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', COMMIT], check=True)
print('repo at', subprocess.run(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'],
                                capture_output=True, text=True).stdout.strip())


In [ ]:
#@title 1 - Provision the Python 3.11 venv with the pinned OpenMMLab stack { display-mode: "form" }
import subprocess, time
from pathlib import Path

REPO = Path('/content/ADA')
VENV = Path('/content/mmrotate_venv')
t0 = time.time()

p = subprocess.Popen(['bash', str(REPO / 'scripts' / 'setup_colab.sh'), str(REPO), str(VENV)],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='', flush=True)
p.wait()
print()
print('setup exit code', p.returncode, 'after', round((time.time()-t0)/60, 1), 'min')
assert p.returncode == 0, 'setup_colab.sh failed - see the output above'
print('venv python:', (VENV / 'bin' / 'python').resolve())


In [ ]:
#@title 2 - Verify the venv sees the GPU, then mount Drive { display-mode: "form" }
import subprocess, json
from pathlib import Path

VENV = Path('/content/mmrotate_venv')
PY = VENV / 'bin' / 'python'

PROBE = """
import torch, mmcv, mmengine, mmdet, os, mmrotate
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| cuda_ver', torch.version.cuda)
print('device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('vram GiB', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1) if torch.cuda.is_available() else 0)
print('mmcv', mmcv.__version__, '| mmengine', mmengine.__version__, '| mmdet', mmdet.__version__)
print('mmrotate', os.path.dirname(mmrotate.__file__))
"""
r = subprocess.run([str(PY), '-c', PROBE], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print('STDERR', r.stderr[-1500:])
assert r.returncode == 0, 'the venv cannot import the stack'

from google.colab import drive
mounted = False
for attempt in range(3):
    try:
        drive.mount('/content/drive', force_remount=True)
        mounted = True
        print('drive mounted on attempt', attempt + 1)
        break
    except Exception as e:
        print('drive mount attempt', attempt + 1, 'failed:', type(e).__name__, e)

root = Path('/content/drive/MyDrive')
print('MyDrive entries:', [p.name for p in root.iterdir()][:12] if root.is_dir() else 'NOT MOUNTED')
